# 0. 라이브러리 호출

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from google.cloud import bigquery

from kiwipiepy import Kiwi

In [3]:
PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

## question (질문)

In [4]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.polls_question`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

    id                 question_text                created_at
0   99            가장 신비한 매력이 있는 사람은? 2023-03-31 15:22:53+00:00
1  100  "이 사람으로 한 번 살아보고 싶다" 하는 사람은? 2023-03-31 15:22:53+00:00
2  101                     미래의 틱톡커는? 2023-03-31 15:22:54+00:00
3  102               여기서 제일 특이한 친구는? 2023-03-31 15:22:54+00:00
4  103               가장 지켜주고 싶은 사람은? 2023-03-31 15:22:55+00:00


In [5]:
question_created = df[['id', 'question_text', 'created_at']].copy()

question_created['created_at_kst'] = (
    pd.to_datetime(
        question_created['created_at'],
        utc=True,
        errors='coerce',
    )
    .dt.tz_convert('Asia/Seoul')
)

question_created['생성일'] = (
    question_created['created_at_kst'].dt.strftime('%Y-%m-%d')
)

In [6]:
daily_question_counts = (
    question_created
    .dropna(subset=['created_at_kst'])
    .groupby('생성일')['id']
    .nunique()
    .reset_index(name='질문 생성 수')
    .sort_values('생성일')
)

display(daily_question_counts)

,생성일,질문 생성 수
0,2023-04-01,227
1,2023-05-02,235
2,2023-05-04,1
3,2023-05-12,91
4,2023-05-15,982
5,2023-06-02,1523
6,2023-06-06,1966


In [7]:
display(question_created[question_created['question_text'] == 'vote']['created_at_kst'].min())
display(question_created[question_created['question_text'] == 'vote']['created_at_kst'].max())

Timestamp('2023-04-01 20:09:15+0900', tz='Asia/Seoul')

Timestamp('2023-06-06 15:15:50+0900', tz='Asia/Seoul')

In [8]:
display(question_created[question_created['question_text'] != 'vote']['created_at_kst'].min())
display(question_created[question_created['question_text'] != 'vote']['created_at_kst'].max())

Timestamp('2023-04-01 00:22:53+0900', tz='Asia/Seoul')

Timestamp('2023-06-06 15:15:52+0900', tz='Asia/Seoul')

* 질문의 생성은 4월 1일 ~ 6월 6일까지 이루어졌습니다. (KST 기준)
* 질문 생성은 오픈 초리보다 6월에 더 많이 생성되었음을 확인할 수 있었습니다.

In [9]:
vote_questions = (
    df.loc[
        df['question_text'].eq('vote'),
        ['id', 'question_text', 'created_at'],
    ]
    .copy()
)

vote_questions['created_at_kst'] = (
    pd.to_datetime(
        vote_questions['created_at'],
        utc=True,
        errors='coerce',
    )
    .dt.tz_convert('Asia/Seoul')
)

vote_questions['생성일'] = (
    vote_questions['created_at_kst'].dt.strftime('%Y-%m-%d')
)

In [10]:
vote_daily_counts = (
    vote_questions
    .dropna(subset=['created_at_kst'])
    .groupby('생성일')['id']
    .nunique()
    .reset_index(name='vote 생성 수')
    .sort_values('생성일')
)

display(vote_daily_counts)

,생성일,vote 생성 수
0,2023-04-01,1
1,2023-05-02,1
2,2023-05-12,1
3,2023-05-15,14
4,2023-06-02,15
5,2023-06-06,24


### `vote` 데이터 처리

- `question_text`가 `vote`인 질문은 특정 시점에만 발생한 데이터가 아니라 **지속적으로 생성되었으며, 신고 기록 등을 고려했을 때 실제 사용자에게 노출되었던 질문으로 파악됩니다.**

- 다만 현재 데이터만으로는 다음 두 가지 가능성을 명확하게 구분하기 어렵습니다.
  - 실제로 `vote`라는 텍스트를 가진 질문이 생성되어 사용자에게 노출된 경우
  - 특정 조건 또는 신고 누적 등의 영향으로 기존 질문의 텍스트가 `vote`로 변경된 경우

- 이를 판단하거나 어느 한쪽으로 추정할 수 있는 **명확한 근거는 현재 데이터에서 확인되지 않았습니다.**

- 본 분석에서는 **질문 텍스트 자체를 활용한 분석을 진행**하기 때문에, `vote` 데이터를 포함하더라도 실제 질문의 내용이나 특성을 해석하기 어렵습니다. 따라서 **텍스트 분석 대상에서는 `vote` 데이터를 제외**하고 분석을 진행합니다.

> **⚠️ 추가 확인 필요**
>
> - 전처리 과정에서 `vote` 질문에 **다수의 신고 기록이 연결되어 있는 사례**가 확인되었습니다.
> - 따라서 일부 `vote` 질문이 오류 또는 테스트 과정에서 생성된 데이터였으며 실제 사용자에게 노출되었을 가능성도 고려할 수 있습니다.
> - 다만 현재 데이터만으로 해당 원인을 확정할 수 없으므로, **`vote` 데이터의 생성 목적과 실제 서비스에서의 처리 방식에 대한 별도 확인이 필요합니다.**

In [11]:
df = df[df['question_text'] != 'vote']

In [12]:
question_created = df[['id', 'question_text', 'created_at']].copy()

question_created['created_at_kst'] = (
    pd.to_datetime(
        question_created['created_at'],
        utc=True,
        errors='coerce',
    )
    .dt.tz_convert('Asia/Seoul')
)

question_created['생성일'] = (
    question_created['created_at_kst'].dt.strftime('%Y-%m-%d')
)

In [13]:
daily_question_counts = (
    question_created
    .dropna(subset=['created_at_kst'])
    .groupby('생성일')['id']
    .nunique()
    .reset_index(name='질문 생성 수')
    .sort_values('생성일')
)

display(daily_question_counts)

,생성일,질문 생성 수
0,2023-04-01,226
1,2023-05-02,234
2,2023-05-04,1
3,2023-05-12,90
4,2023-05-15,968
5,2023-06-02,1508
6,2023-06-06,1942


# 텍스트 분석
* 제공되고 있는 질문의 포함된 텍스트를 확인하여, 워드 클라우드로 표현하고
* 유저의 신고 기록을 기준으로 부정 / 중립 / 긍정의 질문을 나누어서 분류하고 추가로 표현합니다.

In [14]:
question_created

,id,question_text,created_at,created_at_kst,생성일
0,99,가장 신비한 매력이 있는 사람은?,2023-03-31 15:22:53+00:00,2023-04-01 00:22:53+09:00,2023-04-01
1,100,"""이 사람으로 한 번 살아보고 싶다"" 하는 사람은?",2023-03-31 15:22:53+00:00,2023-04-01 00:22:53+09:00,2023-04-01
2,101,미래의 틱톡커는?,2023-03-31 15:22:54+00:00,2023-04-01 00:22:54+09:00,2023-04-01
3,102,여기서 제일 특이한 친구는?,2023-03-31 15:22:54+00:00,2023-04-01 00:22:54+09:00,2023-04-01
4,103,가장 지켜주고 싶은 사람은?,2023-03-31 15:22:55+00:00,2023-04-01 00:22:55+09:00,2023-04-01
...,...,...,...,...,...
5020,5129,나에게 가장 중요한 사람은?,2023-06-06 06:15:52+00:00,2023-06-06 15:15:52+09:00,2023-06-06
5021,5130,오목을 제일 잘 할 것 같은 사람은?,2023-06-06 06:15:52+00:00,2023-06-06 15:15:52+09:00,2023-06-06
5022,5131,가방에서 쓰레기가 안 나올 것 같은 사람은?,2023-06-06 06:15:52+00:00,2023-06-06 15:15:52+09:00,2023-06-06
5023,5132,아무리 많은 숙제도 30분만에 다 끝내버릴 수 있을 것 같은 친구는?,2023-06-06 06:15:52+00:00,2023-06-06 15:15:52+09:00,2023-06-06


In [15]:
# 전처리 수행 대상 테이블 호출
sql2 = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.polls_questionreport`
"""

# 판다스 데이터프레임으로 변환
df2 = client.query(sql2).to_dataframe()

print(df2.head())

      id reason                created_at  question_id  user_id
0   4852  그냥 싫어 2023-05-07 12:14:13+00:00           99   894226
1   4971  그냥 싫어 2023-05-07 13:43:20+00:00           99   906185
2   5389  그냥 싫어 2023-05-08 02:16:54+00:00           99   944035
3   7884  그냥 싫어 2023-05-09 14:19:10+00:00           99   981801
4  11094  그냥 싫어 2023-05-11 13:26:01+00:00           99   887923


In [16]:
df2.value_counts('reason')

reason
그냥 싫어                   28446
나랑 맞지 않는 질문인 것 같음        9541
불쾌한 질문 내용                5386
자꾸 같은 내용의 질문 반복          3202
어떻게 이런 생각을? 이 질문 최고!     1821
한 친구가 질문을 반복적으로 보냄       1701
기타                        480
이 질문은 재미없어요               471
불쾌한 내용이 포함되어 있음           250
오타가 있음                     68
선정적이거나 자극적인 질문             58
Name: count, dtype: int64

### 질문별 신고 원인 평가 기준

- 각 질문의 신고 사유를 확인하여, **신고의 원인이 질문 자체의 내용에 있다고 볼 수 있는지**를 기준으로 `-1`, `0`, `1`의 점수를 부여합니다.
- 현재 점수는 명확한 객관적 기준이 존재하지 않기 때문에, **질문의 내용과 신고 사유를 바탕으로 주관적으로 판단한 탐색적 지표**입니다.

#### 점수 기준

- **`-1점` : 질문 자체에 문제가 있다고 판단되는 경우**
  - 질문의 내용이 사용자에게 불쾌감이나 부정적인 경험을 유발하여 **질문 자체가 신고의 원인이 되었을 가능성이 높은 경우**

- **`0점` : 질문 자체의 문제라고 판단하기 어려운 경우**
  - 신고가 발생했으나, 신고 사유만으로는 **질문 자체가 사용자에게 부정적인 경험을 유발했다고 판단하기 어려운 경우**

- **`1점` : 질문 자체에는 문제가 없다고 판단되는 경우**
  - 신고가 발생했음에도 질문의 내용 자체에는 특별한 문제가 확인되지 않으며, **질문의 의도나 내용이 긍정적이거나 일반적인 상호작용에 해당한다고 판단되는 경우**

> 해당 점수는 질문의 품질을 객관적으로 평가한 절대적인 지표가 아니라, **신고 발생 원인과 질문 내용의 연관성을 탐색하기 위해 임의로 설정한 분석 기준**입니다.

In [17]:
evaluation_score_map = {
    '그냥 싫어': 0,
    '나랑 맞지 않는 질문인 것 같음': 0,
    '불쾌한 질문 내용': -1,
    '자꾸 같은 내용의 질문 반복': 0,
    '어떻게 이런 생각을? 이 질문 최고!': 1,
    '한 친구가 질문을 반복적으로 보냄': 0,
    '기타': 0,
    '이 질문은 재미없어요': -1,
    '불쾌한 내용이 포함되어 있음': -1,
    '오타가 있음': 0,
    '선정적이거나 자극적인 질문': -1,
}

In [18]:
df2['score'] = (
    df2['reason']
    .astype('string')
    .str.strip()
    .map(evaluation_score_map)
)

In [19]:
unmapped_evaluations = (
    df2.loc[
        df2['reason'].notna()
        & df2['score'].isna(),
        'reason',
    ]
    .drop_duplicates()
    .sort_values()
)

display(unmapped_evaluations)

Series([], Name: reason, dtype: str)

In [20]:
df2['score'] = df2['score'].astype('Int64')

In [21]:
df2

,id,reason,created_at,question_id,user_id,score
0,4852,그냥 싫어,2023-05-07 12:14:13+00:00,99,894226,0
1,4971,그냥 싫어,2023-05-07 13:43:20+00:00,99,906185,0
2,5389,그냥 싫어,2023-05-08 02:16:54+00:00,99,944035,0
3,7884,그냥 싫어,2023-05-09 14:19:10+00:00,99,981801,0
4,11094,그냥 싫어,2023-05-11 13:26:01+00:00,99,887923,0
...,...,...,...,...,...,...
51419,52685,한 친구가 질문을 반복적으로 보냄,2023-06-08 06:40:05+00:00,4070,1445152,0
51420,53070,한 친구가 질문을 반복적으로 보냄,2023-06-12 07:25:12+00:00,4097,1470504,0
51421,53565,한 친구가 질문을 반복적으로 보냄,2023-06-21 10:36:19+00:00,4098,1102804,0
51422,53660,한 친구가 질문을 반복적으로 보냄,2023-06-24 05:28:18+00:00,4137,1345338,0


In [22]:
# df2에서 질문 ID별 평가 집계
question_score_by_id = (
    df2
    .groupby('question_id', as_index=False)
    .agg(
        신고횟수=('id', 'nunique'),
        평가점수합계=('score', 'sum'),
    )
)

display(question_score_by_id.head())

,question_id,신고횟수,평가점수합계
0,99,41,6
1,100,36,-1
2,101,67,2
3,102,80,-6
4,103,46,5


In [23]:
# 전체 질문 데이터에 평가 결과 연결
question_created_with_score = (
    question_created
    .merge(
        question_score_by_id,
        left_on='id',
        right_on='question_id',
        how='left',
        validate='one_to_one',
    )
    .drop(columns='question_id')
)

In [24]:
fill_columns = [
    '신고횟수',
    '평가점수합계',
]

question_created_with_score[fill_columns] = (
    question_created_with_score[fill_columns]
    .fillna(0)
    .astype(int)
)

display(question_created_with_score)

,id,question_text,created_at,created_at_kst,생성일,신고횟수,평가점수합계
0,99,가장 신비한 매력이 있는 사람은?,2023-03-31 15:22:53+00:00,2023-04-01 00:22:53+09:00,2023-04-01,41,6
1,100,"""이 사람으로 한 번 살아보고 싶다"" 하는 사람은?",2023-03-31 15:22:53+00:00,2023-04-01 00:22:53+09:00,2023-04-01,36,-1
2,101,미래의 틱톡커는?,2023-03-31 15:22:54+00:00,2023-04-01 00:22:54+09:00,2023-04-01,67,2
3,102,여기서 제일 특이한 친구는?,2023-03-31 15:22:54+00:00,2023-04-01 00:22:54+09:00,2023-04-01,80,-6
4,103,가장 지켜주고 싶은 사람은?,2023-03-31 15:22:55+00:00,2023-04-01 00:22:55+09:00,2023-04-01,46,5
...,...,...,...,...,...,...,...
4964,5129,나에게 가장 중요한 사람은?,2023-06-06 06:15:52+00:00,2023-06-06 15:15:52+09:00,2023-06-06,0,0
4965,5130,오목을 제일 잘 할 것 같은 사람은?,2023-06-06 06:15:52+00:00,2023-06-06 15:15:52+09:00,2023-06-06,0,0
4966,5131,가방에서 쓰레기가 안 나올 것 같은 사람은?,2023-06-06 06:15:52+00:00,2023-06-06 15:15:52+09:00,2023-06-06,0,0
4967,5132,아무리 많은 숙제도 30분만에 다 끝내버릴 수 있을 것 같은 친구는?,2023-06-06 06:15:52+00:00,2023-06-06 15:15:52+09:00,2023-06-06,0,0


In [25]:
question_text_score_summary = (
    question_created_with_score
    .groupby('question_text', as_index=False)
    .agg(
        신고횟수=('신고횟수', 'sum'),
        평가점수합계=('평가점수합계', 'sum'),
    )
)

In [26]:
question_text_score_summary['평가점수평균'] = (
    question_text_score_summary['평가점수합계']
    / question_text_score_summary['신고횟수'].replace(0, np.nan)
)

# 신고가 없는 질문 문구는 평균 0점
question_text_score_summary['평가점수평균'] = (
    question_text_score_summary['평가점수평균']
    .fillna(0)
    .round(2)
)

question_text_score_summary = (
    question_text_score_summary
    .sort_values(
        ['평가점수합계', '신고횟수'],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

display(question_text_score_summary)

,question_text,신고횟수,평가점수합계,평가점수평균
0,마스크가 잘 어울리는 사람은?,988,-317,-0.32
1,발냄새가 호두과자 냄새일 것 같은 사람은?,803,-289,-0.36
2,등빨이 가장 좋은 사람은?,660,-203,-0.31
3,어깨가 가장 넓은 사람은?,533,-146,-0.27
4,먹방을 가장 잘할 것 같은 사람은?,493,-136,-0.28
...,...,...,...,...
3897,사진보다 실물이 더 나은 사람은?,55,15,0.27
3898,항상 좋은 냄새가 나는 사람은?,46,15,0.33
3899,가장 잘생긴 사람은?,140,16,0.11
3900,볼수록 매력있는 사람은?,52,16,0.31


In [27]:
question_text_score_summary.describe()

,신고횟수,평가점수합계,평가점수평균
count,3902.000000,3902.000000,3902.000000
mean,13.054331,-1.105843,-0.052819
std,35.149545,9.692627,0.164172
min,0.000000,-317.000000,-1.000000
25%,1.000000,-1.000000,-0.030000
50%,2.000000,0.000000,0.000000
75%,14.000000,0.000000,0.000000
max,988.000000,21.000000,0.460000


In [28]:
# 신고 횟수 top 10 질문
question_text_score_summary.sort_values(
    by='신고횟수'
    , ascending=False
).head(10)

,question_text,신고횟수,평가점수합계,평가점수평균
0,마스크가 잘 어울리는 사람은?,988,-317,-0.32
1,발냄새가 호두과자 냄새일 것 같은 사람은?,803,-289,-0.36
2,등빨이 가장 좋은 사람은?,660,-203,-0.31
3,어깨가 가장 넓은 사람은?,533,-146,-0.27
4,먹방을 가장 잘할 것 같은 사람은?,493,-136,-0.28
5,설레면 콧구멍이 커지는 친구는?,338,-103,-0.30
8,발냄새가 가장 향긋할 것 같은 사람은?,307,-85,-0.28
12,콧수염을 기르면 잘 어울릴 것 같은 사람은?,283,-61,-0.22
16,먹방 찍으면 100만 유튜버가 될 것 같은 친구,258,-47,-0.18
24,치킨 중독인 것 같은 사람은?,239,-30,-0.13


In [29]:
# 평가 점수 top 10 & bottom 10
display(question_text_score_summary.sort_values(
    by='평가점수합계'
    , ascending=False
).head(10))

display(question_text_score_summary.sort_values(
    by='평가점수합계'
    , ascending=False
).tail(10))

,question_text,신고횟수,평가점수합계,평가점수평균
3901,주변 사람들을 제일 많이 챙겨주는 사람은?,46,21,0.46
3900,볼수록 매력있는 사람은?,52,16,0.31
3899,가장 잘생긴 사람은?,140,16,0.11
3898,항상 좋은 냄새가 나는 사람은?,46,15,0.33
3897,사진보다 실물이 더 나은 사람은?,55,15,0.27
3896,여기서 가장 예쁜 사람은?,67,15,0.22
3895,기쁜 일이 있을때 제일 먼저 알려주고 싶은 사람은?,38,13,0.34
3894,연인에게 가장 잘 대해줄거 같은 사람은?,50,13,0.26
3893,여기서 가장 귀여운 사람은?,54,13,0.24
3888,누가 봐도 좋아할 것 같은 호감인 사람은?,56,12,0.21


,question_text,신고횟수,평가점수합계,평가점수평균
9,첫 키스를 가장 먼저 해봤을 것 같은 사람은?,220,-78,-0.35
8,발냄새가 가장 향긋할 것 같은 사람은?,307,-85,-0.28
7,이모티콘만 보고 알아서 상상해서 골라봐,218,-89,-0.41
6,가장 매력적인 사람은?,196,-96,-0.49
5,설레면 콧구멍이 커지는 친구는?,338,-103,-0.30
4,먹방을 가장 잘할 것 같은 사람은?,493,-136,-0.28
3,어깨가 가장 넓은 사람은?,533,-146,-0.27
2,등빨이 가장 좋은 사람은?,660,-203,-0.31
1,발냄새가 호두과자 냄새일 것 같은 사람은?,803,-289,-0.36
0,마스크가 잘 어울리는 사람은?,988,-317,-0.32


In [30]:
# 평가점수 평균 top 10 & bottom 10
display(question_text_score_summary.sort_values(
    by='평가점수평균'
    , ascending=False
).head(10))

display(question_text_score_summary.sort_values(
    by='평가점수평균'
    , ascending=False
).tail(10))

,question_text,신고횟수,평가점수합계,평가점수평균
3901,주변 사람들을 제일 많이 챙겨주는 사람은?,46,21,0.46
3895,기쁜 일이 있을때 제일 먼저 알려주고 싶은 사람은?,38,13,0.34
3898,항상 좋은 냄새가 나는 사람은?,46,15,0.33
3900,볼수록 매력있는 사람은?,52,16,0.31
3887,힘든 일이 있을때 제일 의지가 될것 같은 사람은?,35,11,0.31
3867,모든 사람과 잘 지낼 것 같은 사람은?,23,7,0.30
3886,미래에 여기서 제일 유명해질거 같은 사람은?,34,10,0.29
3897,사진보다 실물이 더 나은 사람은?,55,15,0.27
3894,연인에게 가장 잘 대해줄거 같은 사람은?,50,13,0.26
3892,같이 있으면 시간 가는줄 모르겠는 사람은?,46,12,0.26


,question_text,신고횟수,평가점수합계,평가점수평균
1009,사달라는거 다 사줄 것 같은 사람은?,1,-1,-1.0
1006,방이 깔끔할 것 같은 친구는?,1,-1,-1.0
325,왜 친해졌는지 모르겠는 사람,3,-3,-1.0
1005,밥 제일 잘 사줄 것 같은 사람,1,-1,-1.0
1004,반응이 로봇 같은 사람은?,1,-1,-1.0
324,24시간 폰만 할 것 같은 사람은?,3,-3,-1.0
1003,미소가 잘 어울리는 친구는?,1,-1,-1.0
1001,매일 매일 기분 좋아보이는 사람,1,-1,-1.0
1000,만능캐인친구,1,-1,-1.0
1002,무인도 한달살기에 데려가고 싶은 친구,1,-1,-1.0


### 전체 워드 클라우드

In [ ]:
kiwi = Kiwi()

In [36]:
stopwords = {
    '사람',
    '친구',
}

In [37]:
def extract_nouns(text):
    return [
        token.form
        for token in kiwi.tokenize(str(text))
        if token.tag in ['NNG', 'NNP']
        and token.form not in stopwords
    ]

In [38]:
noun_check = (
    question_text_score_summary[
        ['question_text']
    ]
    .head(20)
    .copy()
)

noun_check['추출 명사'] = (
    noun_check['question_text']
    .apply(extract_nouns)
)

display(noun_check)

,question_text,추출 명사
0,마스크가 잘 어울리는 사람은?,[마스크]
1,발냄새가 호두과자 냄새일 것 같은 사람은?,"[발, 냄새, 호두, 과자, 냄새]"
2,등빨이 가장 좋은 사람은?,[등빨]
3,어깨가 가장 넓은 사람은?,[어깨]
4,먹방을 가장 잘할 것 같은 사람은?,[먹방]
5,설레면 콧구멍이 커지는 친구는?,[콧구멍]
6,가장 매력적인 사람은?,[매력]
7,이모티콘만 보고 알아서 상상해서 골라봐,"[이모티콘, 상상]"
8,발냄새가 가장 향긋할 것 같은 사람은?,"[발, 냄새]"
9,첫 키스를 가장 먼저 해봤을 것 같은 사람은?,[키스]


In [39]:
from collections import Counter

noun_counts = Counter(
    noun
    for question in question_text_score_summary['question_text']
    for noun in extract_nouns(question)
)

noun_frequency = pd.DataFrame(
    noun_counts.most_common(),
    columns=['명사', '등장 횟수'],
)

display(noun_frequency.head(30))

,명사,등장 횟수
0,때,204
1,말,60
2,학교,57
3,집,49
4,매력,41
5,연락,41
6,노래,40
7,시간,38
8,연애,37
9,나중,37
